# Definition-Based Domination Solver

Choose one of **10 special graph families**, enter its vertex count, and compute a dominating set from **closed neighborhoods / adjacency**, not from a domination-number lookup formula. Optimality is certified by the general lower bound $\lceil n/(\Delta+1)\rceil$.

**Visualization is automatic for $n\le 50$** and never attempted when $n>50$. For large graphs, selected vertices can optionally be written to a text file.

For visualization install `networkx` and `matplotlib` (in a separate cell or terminal): `pip install networkx matplotlib`. All numerical calculations use the standard library.

This tool supports the specified ten special graph families, not arbitrary graphs.

In [1]:
"""Domination number from the DEFINITION, with visualization only for n <= 50.

Supported special graphs: path, cycle, complete, star, wheel, fan,
edgeless, matching (+ one isolate if odd), complete bipartite,
and complete multipartite.

No special-family domination-number formulas are used. The algorithm builds
S, verifies N[S] = V, and certifies optimality using the general degree bound:
    gamma(G) >= ceil(|V| / (Delta(G) + 1)).

Very large graphs use implicit adjacency: no million-node NetworkX graph
or quadratic adjacency matrix is built. This is NOT a general exact solver
for arbitrary input graphs.

Install optional visualization dependencies:
    pip install networkx matplotlib
Run:
    python domination_full_updated.py
"""

from bisect import bisect_right
from collections import deque
from dataclasses import dataclass
from pathlib import Path

VISUAL_LIMIT = 50

GRAPH_TYPES = {
    1: ("path", "Path graph"),
    2: ("cycle", "Cycle graph"),
    3: ("complete", "Complete graph"),
    4: ("star", "Star graph"),
    5: ("wheel", "Wheel graph"),
    6: ("fan", "Fan graph"),
    7: ("edgeless", "Edgeless graph"),
    8: ("matching", "Matching (one isolated vertex if odd)"),
    9: ("bipartite", "Complete bipartite graph"),
    10: ("multipartite", "Complete multipartite graph"),
}
SPARSE_TYPES = {"path", "cycle", "edgeless", "matching"}


@dataclass
class Result:
    count: int
    lower_bound: int
    exact: bool
    first: list[int]     # Zero-based vertex IDs
    last: list[int]
    selected_small: list[int]   # Entire set only when n <= VISUAL_LIMIT


def validate(kind: str, n: int, parts: list[int] | None) -> None:
    if kind not in {item[0] for item in GRAPH_TYPES.values()}:
        raise ValueError("Unsupported graph type")
    if n < 1:
        raise ValueError("The graph must have at least one vertex")
    if kind == "cycle" and n < 3:
        raise ValueError("A cycle graph requires at least 3 vertices")
    if kind == "wheel" and n < 4:
        raise ValueError("A wheel graph requires at least 4 vertices")
    if kind == "fan" and n < 3:
        raise ValueError("A fan graph requires at least 3 vertices")
    if kind in {"bipartite", "multipartite"}:
        if not parts or len(parts) < 2 or any(x <= 0 for x in parts):
            raise ValueError("At least two nonempty parts are required")
        if sum(parts) != n:
            raise ValueError("The part sizes must add up to the vertex count")
        if kind == "bipartite" and len(parts) != 2:
            raise ValueError("Bipartite graphs have exactly two parts")


def sparse_closed_neighbors(kind: str, n: int, v: int) -> tuple[int, ...]:
    """N[v], including v itself. Each neighborhood has at most 3 vertices."""
    if kind == "path":
        return tuple(x for x in (v - 1, v, v + 1) if 0 <= x < n)
    if kind == "cycle":
        return ((v - 1) % n, v, (v + 1) % n)
    if kind == "matching":
        partner = v ^ 1
        return (v, partner) if partner < n else (v,)
    if kind == "edgeless":
        return (v,)
    raise ValueError("Not a sparse graph family")


def greedy_sparse_vertices(kind: str, n: int):
    """Construct a dominating set from neighborhoods, using O(n) memory.

    For each still-undominated vertex u, choose a vertex in N[u] covering
    the most presently undominated vertices. This is not generally exact,
    so optimality is certified separately with a lower bound.
    """
    dominated = bytearray(n)
    covered_count = 0
    for u in range(n):
        if dominated[u]:
            continue
        choices = sparse_closed_neighbors(kind, n, u)
        chosen = max(
            choices,
            key=lambda v: sum(not dominated[w] for w in
                              sparse_closed_neighbors(kind, n, v)),
        )
        for w in sparse_closed_neighbors(kind, n, chosen):
            if not dominated[w]:
                dominated[w] = 1
                covered_count += 1
        yield chosen
    assert covered_count == n, "Constructed set failed to dominate the graph"


def dense_selected_vertices(kind: str, n: int,
                            parts: list[int] | None) -> list[int]:
    """Construct a compact candidate using actual adjacency structure.

    Vertex 0 is universal in complete/star/wheel/fan graphs. In complete
    multipartite graphs, a singleton part gives a universal vertex; if no
    singleton exists, one vertex from each of two parts dominates all parts.
    These are CONSTRUCTIONS, not lookups of domination-number formulas.
    """
    if kind in {"complete", "star", "wheel", "fan"}:
        return [0]
    assert parts is not None
    start = 0
    for size in parts:
        if size == 1:
            return [start]
        start += size
    return [0, parts[0]]


def are_adjacent(kind: str, n: int, u: int, v: int,
                 ends: list[int] | None = None) -> bool:
    """Adjacency oracle: no edge list or adjacency matrix is necessary."""
    if u == v:
        return False
    if kind == "path":
        return abs(u - v) == 1
    if kind == "cycle":
        return abs(u - v) in (1, n - 1)
    if kind == "matching":
        return u // 2 == v // 2 and min(u, v) % 2 == 0
    if kind == "edgeless":
        return False
    if kind == "complete":
        return True
    if kind == "star":
        return u == 0 or v == 0
    if kind in {"wheel", "fan"}:
        if u == 0 or v == 0:
            return True
        if abs(u - v) == 1:
            return True
        return kind == "wheel" and abs(u - v) == n - 2
    if kind in {"bipartite", "multipartite"}:
        assert ends is not None
        return bisect_right(ends, u) != bisect_right(ends, v)
    raise ValueError("Unknown graph family")


def verify_dense(kind: str, n: int, chosen: list[int],
                 parts: list[int] | None) -> None:
    """Check definition N[S]=V, even for n=1,000,000 (O(n |S|))."""
    ends = None
    if parts is not None:
        ends = []
        total = 0
        for size in parts:
            total += size
            ends.append(total)
    chosen_set = set(chosen)
    for v in range(n):
        if v not in chosen_set and not any(
            are_adjacent(kind, n, u, v, ends) for u in chosen
        ):
            raise AssertionError(f"Vertex {v + 1} is not dominated")


def maximum_degree(kind: str, n: int,
                   parts: list[int] | None) -> int:
    """Degree is used for a GENERAL lower bound, not a gamma formula."""
    if kind == "path":
        return min(2, n - 1)
    if kind == "cycle":
        return 2
    if kind == "matching":
        return 1 if n >= 2 else 0
    if kind == "edgeless":
        return 0
    if kind in {"complete", "star", "wheel", "fan"}:
        return n - 1
    assert parts is not None
    return n - min(parts)


def calculate(kind: str, n: int,
              parts: list[int] | None = None) -> Result:
    validate(kind, n, parts)
    first: list[int] = []
    last: deque[int] = deque(maxlen=5)
    selected_small: list[int] = []
    count = 0

    if kind in SPARSE_TYPES:
        candidates = greedy_sparse_vertices(kind, n)
    else:
        candidates = dense_selected_vertices(kind, n, parts)
        verify_dense(kind, n, candidates, parts)

    for v in candidates:
        count += 1
        if len(first) < 10:
            first.append(v)
        last.append(v)
        if n <= VISUAL_LIMIT:
            selected_small.append(v)

    delta = maximum_degree(kind, n, parts)
    lower_bound = (n + delta) // (delta + 1)  # ceil(n/(delta+1))
    return Result(count, lower_bound, count == lower_bound,
                  first, list(last), selected_small)


def vertex_label(v: int) -> str:
    """0 -> A, 25 -> Z, 26 -> AA, ... for the small visualization."""
    result = ""
    while True:
        v, r = divmod(v, 26)
        result = chr(65 + r) + result
        if v == 0:
            return result
        v -= 1


def plot_graph(kind: str, n: int, selected: list[int],
              parts: list[int] | None = None) -> None:
    """Draw only small graphs. Never create 50+ vertex graph drawings."""
    if n > VISUAL_LIMIT:
        print(f"Visualization skipped: {n:,} > {VISUAL_LIMIT} vertices.")
        return
    try:
        import matplotlib.pyplot as plt
        import networkx as nx
        from matplotlib.lines import Line2D
    except ImportError:
        print("Visualization unavailable. Install: pip install networkx matplotlib")
        return

    ends = None
    if parts is not None:
        ends = []
        total = 0
        for size in parts:
            total += size
            ends.append(total)
    graph = nx.Graph()
    graph.add_nodes_from(range(n))
    for u in range(n):
        for v in range(u + 1, n):
            if are_adjacent(kind, n, u, v, ends):
                graph.add_edge(u, v)

    if kind == "path":
        pos = {v: (v, 0) for v in graph.nodes}
    elif kind == "cycle":
        pos = nx.circular_layout(graph)
    elif kind in {"star", "wheel", "fan"}:
        rim = list(range(1, n))
        pos = nx.circular_layout(rim)
        pos[0] = (0, 0)
    elif kind == "matching":
        pos = {v: (v // 2, v % 2) for v in graph.nodes}
    elif kind in {"bipartite", "multipartite"}:
        for v in range(n):
            graph.nodes[v]["subset"] = bisect_right(ends, v)
        pos = nx.multipartite_layout(graph, subset_key="subset")
    elif kind == "complete":
        pos = nx.circular_layout(graph)
    else:
        pos = nx.spring_layout(graph, seed=42)

    fig, ax = plt.subplots(figsize=(16, 6.5) if kind in {"path", "matching"}
                            else (10, 8))
    chosen = set(selected)
    nx.draw_networkx_edges(graph, pos, ax=ax, edge_color="#64748b",
                           alpha=0.22 if graph.number_of_edges() > 300 else 0.55,
                           width=0.9)
    nx.draw_networkx_nodes(graph, pos, ax=ax, nodelist=list(graph.nodes),
                           node_color=["#dc3545" if v in chosen else "#3b82f6"
                                       for v in graph.nodes],
                           node_size=470 if n <= 25 else 300,
                           linewidths=0.8, edgecolors="white")
    nx.draw_networkx_labels(graph, pos, ax=ax,
                            labels={v: vertex_label(v) for v in graph.nodes},
                            font_color="white", font_weight="bold",
                            font_size=9 if n <= 25 else 7)
    ax.set_title(
    f"{kind.title()} Graph | Selected dominating vertices: {len(selected)}\n"
    "Coded by: J.K Padro, MS Math (CAR)",
    fontsize=13,
    fontweight="bold",
    linespacing=1.8,
    pad=20
)
    ax.legend(handles=[
        Line2D([0], [0], marker="o", color="w", label="Dominating set",
               markerfacecolor="#dc3545", markersize=10),
        Line2D([0], [0], marker="o", color="w", label="Other vertices",
               markerfacecolor="#3b82f6", markersize=10),
    ], loc="best")
    ax.axis("off")
    fig.tight_layout()
    plt.show()
    plt.close(fig)


def get_positive_int(prompt: str, maximum: int | None = None) -> int:
    while True:
        try:
            n = int(input(prompt).strip().replace(",", ""))
            if n >= 1 and (maximum is None or n <= maximum):
                return n
        except ValueError:
            pass
        print("Please enter a positive whole number"
              + (f" at most {maximum}." if maximum is not None else "."))


def save_selected_set(kind: str, n: int, parts: list[int] | None) -> None:
    """Regenerate on demand; avoid storing a million selected Python ints."""
    out = Path(f"minimum_dominating_set_{kind}_{n}.txt")
    if kind in SPARSE_TYPES:
        selected = greedy_sparse_vertices(kind, n)
    else:
        selected = dense_selected_vertices(kind, n, parts)
    with out.open("w", encoding="utf-8") as file:
        for v in selected:
            file.write(f"{v + 1}\n")
    print(f"Full selected set saved to: {out.resolve()}")


def domination_number() -> None:
    print("\nDOMINATION NUMBER SOLVER (DEFINITION-BASED)")
    print(f"Visualizes at most {VISUAL_LIMIT} vertices.\n")
    for number, (_, description) in GRAPH_TYPES.items():
        print(f"{number:2}. {description}")

    option = get_positive_int("\nChoose graph type (1-10): ", maximum=10)
    kind, title = GRAPH_TYPES[option]
    n = get_positive_int("Enter the number of vertices: ")
    parts = None

    if kind == "bipartite":
        if n < 2:
            print("Complete bipartite graphs require at least two vertices.")
            return
        a = get_positive_int("Number of vertices in part A: ", maximum=n - 1)
        parts = [a, n - a]
    elif kind == "multipartite":
        while True:
            raw = input("Part sizes separated by spaces (must sum to n): ")
            try:
                parts = [int(x) for x in raw.split()]
                validate(kind, n, parts)
                break
            except ValueError as exc:
                print(f"Invalid parts: {exc}")

    try:
        result = calculate(kind, n, parts)
    except ValueError as exc:
        print("Input error:", exc)
        return

    print("\n--- RESULTS ---")
    print(f"Graph type: {title}")
    print(f"Number of vertices: {n:,}")
    print(f"Size of constructed dominating set: {result.count:,}")
    print(f"General lower bound: {result.lower_bound:,}")
    if result.exact:
        print(f"EXACT domination number gamma(G): {result.count:,}")
        print("Proof: N[S] = V and |S| reaches the degree lower bound.")
    else:
        print("This is an upper bound, NOT a proven domination number.")

    if n <= VISUAL_LIMIT:
        print("Selected vertices:",
              [vertex_label(v) for v in result.selected_small])
        print("Displaying graph (red = selected, blue = not selected)...")
        plot_graph(kind, n, result.selected_small, parts)
    else:
        print("First selected vertex indices:", [v + 1 for v in result.first])
        print("Last selected vertex indices:", [v + 1 for v in result.last])
        print(f"Visualization skipped: vertex count exceeds {VISUAL_LIMIT}.")
        if input("Save entire selected set to a text file? (y/N): ").strip().lower() in {"y", "yes"}:
            save_selected_set(kind, n, parts)




## Start interactive solver
Run the cell below. Type your graph type and number of vertices at the prompts. For bipartite and multipartite graphs, provide the part sizes too.

In [5]:
domination_number()


DOMINATION NUMBER SOLVER (DEFINITION-BASED)
Visualizes at most 50 vertices.

 1. Path graph
 2. Cycle graph
 3. Complete graph
 4. Star graph
 5. Wheel graph
 6. Fan graph
 7. Edgeless graph
 8. Matching (one isolated vertex if odd)
 9. Complete bipartite graph
10. Complete multipartite graph



Choose graph type (1-10):  2
Enter the number of vertices:  1000000



--- RESULTS ---
Graph type: Cycle graph
Number of vertices: 1,000,000
Size of constructed dominating set: 333,334
General lower bound: 333,334
EXACT domination number gamma(G): 333,334
Proof: N[S] = V and |S| reaches the degree lower bound.
First selected vertex indices: [1000000, 3, 6, 9, 12, 15, 18, 21, 24, 27]
Last selected vertex indices: [999987, 999990, 999993, 999996, 999997]
Visualization skipped: vertex count exceeds 50.


Save entire selected set to a text file? (y/N):  N
